In [1]:
# ─────────────────────────────────────────────────────────────
# 01 DATA LOADING — LANL Authentication Dataset
# Loads the curated parsed_logs.csv (68,221 events, 14.98% attack)
# built from LANL Comprehensive Multi-Source Cyber-Security Events
# ─────────────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv('parsed_logs.csv')

print(f"Loaded parsed_logs.csv")
print(f"Total events: {len(df):,}")
print(f"Attack events: {df['is_attack'].sum():,} ({df['is_attack'].mean()*100:.2f}%)")
print(f"Normal events: {(df['is_attack']==0).sum():,}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

Loaded parsed_logs.csv
Total events: 68,221
Attack events: 10,221 (14.98%)
Normal events: 58,000

Columns: ['time', 'source_user', 'destination_user', 'source_computer', 'destination_computer', 'authentication_type', 'logon_type', 'authentication_orientation', 'success_failure', 'is_attack']

First 5 rows:


,time,source_user,destination_user,source_computer,destination_computer,authentication_type,logon_type,authentication_orientation,success_failure,is_attack
0,1036800,C11731$@DOM1,C11731$@DOM1,C11731,C11731,?,Network,LogOff,Success,0
1,1036800,C5396$@DOM1,C5396$@DOM1,C457,C457,?,Network,LogOff,Success,0
2,1036801,C14431$@DOM1,C14431$@DOM1,C1065,C1065,?,Network,LogOff,Success,0
3,1036801,C1484$@DOM1,U198@DOM1,C1484,C1484,Negotiate,Batch,LogOn,Success,0
4,1036801,U198@DOM1,U198@DOM1,C1484,C1484,?,?,TGT,Success,0


In [2]:
# ─────────────────────────────────────────────────────────────
# OBJECTIVE 1 — MongoDB Ingestion Pipeline
# Stores parsed LANL authentication events in MongoDB for
# scalable log storage and correlation (production-ready architecture)
# ─────────────────────────────────────────────────────────────
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
import pandas as pd

df_mongo = pd.read_csv('parsed_logs.csv')

try:
    client = MongoClient('mongodb://localhost:27017/',
                         serverSelectionTimeoutMS=5000)
    client.server_info()

    db         = client['soc_auth_logs_lanl']
    collection = db['lanl_auth_events']

    collection.drop()

    records = df_mongo.to_dict('records')
    result  = collection.insert_many(records)

    print(f"MongoDB Pipeline — Objective 1 (LANL Dataset)")
    print("=" * 50)
    print(f"Database   : soc_auth_logs_lanl")
    print(f"Collection : lanl_auth_events")
    print(f"Inserted   : {len(result.inserted_ids):,} documents")

    # ── Verify with queries ──────────────────────────────────
    total = collection.count_documents({})
    print(f"Verified   : {total:,} documents in collection")

    # Query 1: Count attack events
    attack_count = collection.count_documents({'is_attack': 1})
    print(f"\nAttack events     : {attack_count:,}")

    # Query 2: Count failed logins (genuine field this time)
    failed_count = collection.count_documents({'success_failure': 'Fail'})
    print(f"Failed login events: {failed_count:,}")

    # Query 3: Count distinct source users
    distinct_users = len(collection.distinct('source_user'))
    print(f"Distinct source users: {distinct_users:,}")

    # Query 4: Sample document
    sample = collection.find_one({'is_attack': 1})
    print(f"\nSample document (attack event):")
    print(f"  source_user  : {sample.get('source_user')}")
    print(f"  dest_computer: {sample.get('destination_computer')}")
    print(f"  logon_type   : {sample.get('logon_type')}")
    print(f"  success_failure: {sample.get('success_failure')}")
    print(f"  time         : {sample.get('time')}")

    # ── Create indexes for fast querying ─────────────────────
    collection.create_index('source_user')
    collection.create_index('time')
    collection.create_index('is_attack')
    collection.create_index('success_failure')
    print(f"\nIndexes created on: source_user, time, is_attack, success_failure")
    print(f"\nObjective 1 COMPLETE — MongoDB pipeline operational")

    client.close()

except ConnectionFailure as e:
    print(f"MongoDB connection failed: {e}")
    print("Ensure MongoDB is installed and running as a service.")
    print("Download from: mongodb.com/try/download/community")
except Exception as e:
    print(f"Error: {e}")

MongoDB Pipeline — Objective 1 (LANL Dataset)
Database   : soc_auth_logs_lanl
Collection : lanl_auth_events
Inserted   : 68,221 documents
Verified   : 68,221 documents in collection

Attack events     : 10,221
Failed login events: 591
Distinct source users: 14,309

Sample document (attack event):
  source_user  : U86@DOM1
  dest_computer: C561
  logon_type   : Network
  success_failure: Success
  time         : 1036803

Indexes created on: source_user, time, is_attack, success_failure

Objective 1 COMPLETE — MongoDB pipeline operational


In [4]:
# ─────────────────────────────────────────────────────────────
# OBJECTIVE 1 — Elasticsearch Indexing Pipeline
# Indexes parsed LANL authentication events into Elasticsearch
# for SIEM rule deployment and Kibana visualisation
# ─────────────────────────────────────────────────────────────
from elasticsearch import Elasticsearch, helpers
import pandas as pd

df_es = pd.read_csv('parsed_logs.csv')

try:
    es = Elasticsearch(
        "http://localhost:9200",
        request_timeout=30
    )

    if not es.ping():
        raise ConnectionError("Could not connect to Elasticsearch")

    print(f"Elasticsearch Pipeline — Objective 1 (LANL Dataset)")
    print("=" * 50)

    index_name = "lanl-auth-logs"

    # Delete existing index for a clean run
    if es.indices.exists(index=index_name):
        es.indices.delete(index=index_name)
        print(f"Deleted existing index: {index_name}")

    # Create index with explicit mapping
    mapping = {
        "mappings": {
            "properties": {
                "time": {"type": "long"},
                "source_user": {"type": "keyword"},
                "destination_user": {"type": "keyword"},
                "source_computer": {"type": "keyword"},
                "destination_computer": {"type": "keyword"},
                "authentication_type": {"type": "keyword"},
                "logon_type": {"type": "keyword"},
                "authentication_orientation": {"type": "keyword"},
                "success_failure": {"type": "keyword"},
                "is_attack": {"type": "integer"}
            }
        }
    }
    es.indices.create(index=index_name, body=mapping)
    print(f"Created index: {index_name}")

    # Bulk index all documents
    records = df_es.to_dict('records')
    actions = [
        {"_index": index_name, "_source": record}
        for record in records
    ]

    success_count, errors = helpers.bulk(es, actions, raise_on_error=False)
    print(f"\nIndexed   : {success_count:,} documents")
    print(f"Errors    : {len(errors) if errors else 0}")

    # Refresh to make documents searchable immediately
    es.indices.refresh(index=index_name)

    # Verify
    count = es.count(index=index_name)['count']
    print(f"Verified  : {count:,} documents in index")

    # Query 1: Count attack events
    attack_query = {"query": {"term": {"is_attack": 1}}}
    attack_count = es.count(index=index_name, body=attack_query)['count']
    print(f"\nAttack events in index: {attack_count:,}")

    # Query 2: Count failed logins
    fail_query = {"query": {"term": {"success_failure": "Fail"}}}
    fail_count = es.count(index=index_name, body=fail_query)['count']
    print(f"Failed login events in index: {fail_count:,}")

    print(f"\nObjective 1 COMPLETE — Elasticsearch pipeline operational")
    print(f"Index '{index_name}' ready for SIEM rule deployment in Kibana")

except ConnectionError as e:
    print(f"Elasticsearch connection failed: {e}")
    print("Ensure Elasticsearch is installed and running.")
except Exception as e:
    print(f"Error: {e}")

Elasticsearch Pipeline — Objective 1 (LANL Dataset)
Created index: lanl-auth-logs

Indexed   : 68,221 documents
Errors    : 0
Verified  : 68,221 documents in index

Attack events in index: 10,221
Failed login events in index: 591

Objective 1 COMPLETE — Elasticsearch pipeline operational
Index 'lanl-auth-logs' ready for SIEM rule deployment in Kibana


## Objective 1 — Data Ingestion Pipeline (LANL Dataset)

**Status: COMPLETE**

Source: LANL Comprehensive, Multi-Source Cyber-Security Events (Kent, 2015)
Slice: Days 12-15 (densest red-team attack window), source_user-filtered, capped at 200 events/user

| Metric | Value |
|---|---|
| Total events | 68,221 |
| Attack events | 10,221 (14.98%) |
| Normal events | 58,000 |
| Genuine failed logins | 591 |
| Distinct source users | 14,309 |
| MongoDB collection | soc_auth_logs_lanl.lanl_auth_events |
| Elasticsearch index | lanl-auth-logs |

This dataset was validated to have zero EventID/channel confound between attack and normal populations (unlike the OTRF Mordor dataset used in earlier project iterations), confirmed via authentication_type and logon_type overlap analysis.